# 07 — missing job header

Regression: job `4428420015`, capture **5143**. Reference: [05_job_header.ipynb](05_job_header.ipynb).

Run from repo root.

## load search-results row

In [16]:
from urllib.parse import parse_qs, urlparse

from sqlalchemy import text

from core.db import SessionLocal

JOB_ID = "4428420015"
SEARCH_RESULTS_PATH = "/flagship-web/jobs/search-results/"

query = text("""
    SELECT id, request_url, request_body, response_body, captured_at_ms
    FROM mitm_http_captures
    WHERE request_url LIKE :url_pattern
      AND response_body IS NOT NULL
    ORDER BY captured_at_ms DESC
    LIMIT 50
""")

with SessionLocal() as session:
    candidates = session.execute(
        query,
        {"url_pattern": f"%{SEARCH_RESULTS_PATH}%currentJobId={JOB_ID}%"},
    ).fetchall()

row = None
for candidate in candidates:
    request_url = candidate[1]
    if urlparse(request_url).path != SEARCH_RESULTS_PATH:
        continue
    current_job_id = parse_qs(urlparse(request_url).query).get("currentJobId", [None])[
        0
    ]
    if current_job_id == JOB_ID:
        row = candidate
        break

if row is None:
    raise SystemExit(f"no search-results capture found for job {JOB_ID}")

capture_id, request_url, request_body, response_body, captured_at_ms = row
print("job_id:", JOB_ID)
print("capture_id:", capture_id)
print("captured_at_ms:", captured_at_ms)
print("path:", urlparse(request_url).path)
print("currentJobId:", parse_qs(urlparse(request_url).query)["currentJobId"][0])
print("response chars:", len(response_body))

job_id: 4428420015
capture_id: 5143
captured_at_ms: 1782914678723
path: /flagship-web/jobs/search-results/
currentJobId: 4428420015
response chars: 408973


## locate header strings

In [17]:
from adapters.linkedin import rsc

HEADER_TERMS = [
    "South Australia Police",
    "Data Analyst",
    "Adelaide, South Australia, Australia",
    "2 weeks ago",
    "Over 100 people clicked apply",
    # "Promoted by hirer",
    "Responses managed off LinkedIn",
    # "Full-time",
    # "Apply",
    # "Save",
]

chunks = rsc.parse_stream(response_body)
print(len(chunks), "chunks")

for term in HEADER_TERMS:
    hits = [chunk_id for chunk_id, data in chunks.items() if term in data]
    suffix = f" ... ({len(hits)} total)" if len(hits) > 8 else ""
    print(f"{term!r}: {hits[:8]}{suffix}")

291 chunks
'South Australia Police': ['0', '28', '29', '30', '56', 'd2', 'b2', 'd8'] ... (13 total)
'Data Analyst': ['0', '28', '29', '30', '56', 'd2', 'd8', 'df'] ... (9 total)
'Adelaide, South Australia, Australia': ['28', '29', '30', '56']
'2 weeks ago': ['28', '29']
'Over 100 people clicked apply': ['28', '29']
'Responses managed off LinkedIn': ['28', '29']


## build module lookup

In [18]:
import re

MODULE_RE = re.compile(r'^I\["[^"]*",\[\],"([^"]+)"\]$')


def build_module_lookup(chunks: dict[str, str]) -> dict[str, str]:
    lookup = {}

    for chunk_id, value in chunks.items():
        match = MODULE_RE.match(value)

        if match:
            lookup[chunk_id] = match.group(1)

    return lookup

In [19]:
module_lookup = build_module_lookup(chunks)

## chunk 29

In [20]:
from adapters.linkedin import rsc

parsed = rsc.get_chunk_parsed(response_body, "29")

## component tree

In [21]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Optional


@dataclass
class Node:
    kind: str  # "div", "TrackingScopeProvider", "str", "ref($1)", ...
    value: str | None = None  # only used for string nodes
    parent: Optional["Node"] = None
    children: list["Node"] = field(default_factory=list)

    def add(self, child: "Node"):
        child.parent = self
        self.children.append(child)

    @property
    def path(self):
        node = self
        path = []

        while node.parent is not None:
            path.append(node)
            node = node.parent

        path.reverse()
        return path

    def pretty(self, depth=0):
        indent = "  " * depth

        if self.kind == "str":
            print(f"{indent}{self.value!r}")
        else:
            print(f"{indent}{self.kind}")

        for child in self.children:
            child.pretty(depth + 1)


class Tree:
    def __init__(self, root):
        self.root = root

    def walk(self):
        yield from self._walk(self.root)

    def _walk(self, node):
        yield node
        for child in node.children:
            yield from self._walk(child)

    def find_strings(self, text):
        for node in self.walk():
            if node.kind == "str" and node.value == text:
                yield node

    def strings(self):
        for node in self.walk():
            if node.kind == "str":
                yield node


class Builder:
    def __init__(self, module_lookup):
        self.module_lookup = module_lookup

    def build(self, parsed):

        root = Node("ROOT")
        self._visit(parsed, root)
        return Tree(root)

    def _visit(self, obj, parent):

        if isinstance(obj, dict):
            if "children" in obj:
                children = obj["children"]

                if isinstance(children, list):
                    for child in children:
                        self._visit(child, parent)

                elif isinstance(children, str):
                    parent.add(Node("str", children))

            elif "textProps" in obj:
                self._visit(obj["textProps"], parent)

            return

        if isinstance(obj, list):
            # RSC Element
            if len(obj) == 4 and obj[0] == "$":
                tag = obj[1]

                if tag.startswith("$L"):
                    kind = self.module_lookup.get(tag[2:], tag)

                elif tag.startswith("$"):
                    kind = f"ref({tag})"

                else:
                    kind = f"RSCNode({tag})"

                node = Node(kind)
                parent.add(node)

                self._visit(obj[3], node)
                return

            for child in obj:
                self._visit(child, parent)

            return

        if isinstance(obj, str):
            if obj.startswith("$"):
                return

            parent.add(Node("str", obj))

## chunk 28 tree

In [22]:
from adapters.linkedin import rsc

builder = Builder(module_lookup)
parsed = rsc.get_chunk_parsed(response_body, "28")
tree = builder.build(parsed)
# tree.root.pretty()
len(list(tree.find_strings("South Australia Police")))

1

## extract header

In [23]:
from dataclasses import dataclass


@dataclass
class JobHeader:
    company: str | None = None
    title: str | None = None
    location: str | None = None
    posted: str | None = None
    apply_count: str | None = None
    promoted: bool = False
    hiring_insights: str | None = None


def extract_job_header(tree) -> JobHeader:
    values = [
        text
        for s in tree.strings()
        if (text := s.value.strip())
        and text not in {"div", "p", "span", "·"}
        and not text.startswith("$")
    ]
    promoted = True if values[5] == "Promoted by hirer ·" else False
    if not promoted:
        hiring_insights = values[5]
    else:
        hiring_insights = values[6]
    return JobHeader(
        company=values[0],
        title=values[1],
        location=values[2],
        posted=values[3],
        apply_count=values[4],
        promoted=promoted,
        hiring_insights=hiring_insights,
    )


extract_job_header(tree)

JobHeader(company='South Australia Police', title='Data Analyst', location='Adelaide, South Australia, Australia', posted='2 weeks ago', apply_count='Over 100 people clicked apply', promoted=True, hiring_insights='Responses managed off LinkedIn')

## batch smoke test

In [27]:
from urllib.parse import parse_qs, urlparse

from sqlalchemy import text

from adapters.linkedin import rsc
from core.db import SessionLocal

SEARCH_RESULTS_PATH = "/flagship-web/jobs/search-results/"

query = text("""
    SELECT id, request_url, request_body, response_body, captured_at_ms
    FROM mitm_http_captures
    WHERE request_url LIKE :url_pattern
      AND response_body IS NOT NULL
    ORDER BY captured_at_ms DESC
    LIMIT 50
""")

with SessionLocal() as session:
    candidates = session.execute(
        query,
        {"url_pattern": f"%{SEARCH_RESULTS_PATH}%currentJobId=%"},
    ).fetchall()


for row in candidates:
    request_url = row[1]
    if urlparse(request_url).path != SEARCH_RESULTS_PATH:
        continue
    current_job_id = parse_qs(urlparse(request_url).query).get("currentJobId", [None])[
        0
    ]
    capture_id, request_url, request_body, response_body, captured_at_ms = row

    module_lookup = build_module_lookup(chunks)
    builder = Builder(module_lookup)
    parsed = rsc.get_chunk_parsed(response_body, "28")
    tree = builder.build(parsed)

    header = extract_job_header(tree)
    print(header.title)
    # print('-' * 40)

Data Analyst
Data Analyst Food Menu & Design
Data Analyst
Data Analyst
Data Analyst
Senior Data Engineer
Principal Data Engineer
Specialist Director - Snowflake Data Architect | Sydney
Principal Solution Engineer
Data Engineer - Remote (ANZ)
Snowflake Data Engineer - $190k + super - Global Consultancy
Snowflake Architect
Supply Chain Inventory Data Analyst
Associate Managing Consultant, Strategy and Transformation - Generative AI Engineer / Generative AI Data Scientist
Data Cloud Engineer
Customer Data Engineer
Software Engineer (Data)
Data Platform Engineer
Data Analyst - Ancestry & Health Genomics Lab
Data Engineer (12-month MTC)
Software Engineer
Python Data Engineer
Data Engineer
ETL Specialist
Senior Data Engineer
Software Engineer (Data)
Data Engineer
Data Engineer
AI Engineer
Data Engineer (Agentic AI & Azure Cloud)
Data Analyst
Data Analyst
AI Engineer - Melbourne
Data Engineer
Data Scientist
Data - Site Reliability Engineer
Senior Data Analyst
Quantitative Developer
Data Speci